# AIC 2026 - SigLIP2 Feature Extraction Pipeline

Notebook này được thiết kế để chạy trên **Google Colab (GPU T4 miễn phí)** nhằm trích xuất vector đặc trưng từ mô hình `google/siglip2-base-patch16-224` cho bộ dữ liệu AIC. 

**Quy trình hoạt động:**
1. Kết nối Google Drive để lưu file `.npy` vĩnh viễn.
2. Tải và giải nén từng file ZIP Keyframes từ máy chủ BTC.
3. Dùng SigLIP2 trích xuất vector, **L2 Normalize**, và lưu thành `.npy` vào Drive.
4. Xóa file ZIP và ảnh tạm để giải phóng dung lượng, chuyển sang file ZIP tiếp theo.

In [ ]:
!pip install -q transformers torch torchvision pillow requests tqdm

In [ ]:
import os
import glob
import json
import torch
import numpy as np
import requests
import zipfile
import shutil
import torch.nn.functional as F
from PIL import Image
from tqdm.notebook import tqdm
from transformers import AutoProcessor, AutoModel
from google.colab import drive

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# Thư mục lưu file Numpy trên Drive
OUTPUT_DIR = "/content/drive/MyDrive/AIC2026_SigLIP2_Features"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

In [ ]:
# 2. Khởi tạo Mô hình SigLIP2 (English-only, dim=768)
MODEL_ID = "google/siglip2-base-patch16-224"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
model.eval()

In [ ]:
# 3. Danh sách các file ZIP từ BTC CDN
# (Lưu ý: Bạn có thể thêm bớt các link tùy theo tiến độ của mình)
ZIP_LINKS = [
    "https://aic-data.ledo.io.vn/Keyframes_L21.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L22.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L23.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L24.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L25.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_a.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_b.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_c.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_d.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_e.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L27.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L28.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L29.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L30.zip"
]

In [ ]:
# 4. Hàm trích xuất đặc trưng cho 1 Video
def extract_video_features(video_dir, video_id):
    output_path = os.path.join(OUTPUT_DIR, f"{video_id}.npy")
    if os.path.exists(output_path):
        return  # Bỏ qua nếu đã extract xong (Resume an toàn)
        
    # Lấy danh sách ảnh (sắp xếp chuẩn theo số)
    img_paths = sorted(glob.glob(os.path.join(video_dir, "*.jpg")))
    if not img_paths:
        return
        
    batch_size = 64  # GPU T4 dư sức gánh batch 64
    all_embeddings = []
    
    with torch.no_grad():
        for i in range(0, len(img_paths), batch_size):
            batch_paths = img_paths[i:i+batch_size]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            
            inputs = processor(images=images, return_tensors="pt").to(device)
            image_features = model.get_image_features(**inputs)
            
            # L2 NORMALIZATION (QUAN TRỌNG CHO FAISS IndexFlatIP)
            image_features = F.normalize(image_features, p=2, dim=-1)
            
            all_embeddings.append(image_features.cpu().numpy().astype(np.float32))
            
    if all_embeddings:
        video_embeddings = np.vstack(all_embeddings)
        np.save(output_path, video_embeddings)

# 5. Vòng lặp tải, xử lý và dọn dẹp
for url in ZIP_LINKS:
    zip_name = url.split("/")[-1]
    zip_path = os.path.join("/content", zip_name)
    extract_path = os.path.join("/content", "temp_extract")
    
    print(f"\nProcessing {zip_name}...")
    
    # Tải file (Streaming download)
    if not os.path.exists(zip_path):
        response = requests.get(url, stream=True)
        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                
    # Giải nén
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
        
    # Tìm tất cả các thư mục Video (Lxx_Vyyy)
    video_dirs = [d for d in glob.glob(os.path.join(extract_path, "**", "*"), recursive=True) if os.path.isdir(d)]
    video_dirs = [d for d in video_dirs if len(glob.glob(os.path.join(d, "*.jpg"))) > 0]
    
    print(f"Found {len(video_dirs)} videos in {zip_name}. Extracting features...")
    
    # Xử lý từng video
    for v_dir in tqdm(video_dirs):
        video_id = os.path.basename(v_dir)
        extract_video_features(v_dir, video_id)
        
    # Dọn dẹp để không tràn disk Colab
    shutil.rmtree(extract_path)
    os.remove(zip_path)
    print(f"Cleaned up {zip_name}.")

print("\nALL DONE! Features saved to Google Drive.")